In [1]:
def neg_to_zero(l):
    for i in range(len(l)):
        if l[i] < 0:
            l[i] = 0

    return l

print(neg_to_zero([3, 3, -1, -5, 2, 4]))


[3, 3, 0, 0, 2, 4]


In [2]:
# @title Data Loader

import os

try:
    import google.colab
    REPO_URL = "https://github.com/wtheisen/nd-cse-10124-lectures.git"

    REPO_NAME = "/content/nd-cse-10124-lectures"
    L_PATH = "nd-cse-10124-lectures"

    %cd /content/
    !rm -r {REPO_NAME}

    # Clone repo
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}

        # cd into the data folder
        %cd {L_PATH}
        !pwd

except ImportError:
    print("Unable to download repo, either:")
    print("\tA.) You're not on colab")
    print("\tB.) It has already been cloned")

!pwd
import irishGPT as iGPT

/content
rm: cannot remove '/content/nd-cse-10124-lectures': No such file or directory
Cloning into 'nd-cse-10124-lectures'...
remote: Enumerating objects: 354, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 354 (delta 46), reused 69 (delta 31), pack-reused 267 (from 1)
Receiving objects: 100% (354/354), 33.92 MiB | 16.45 MiB/s, done.
Resolving deltas: 100% (222/222), done.
/content/nd-cse-10124-lectures
/content/nd-cse-10124-lectures
/content/nd-cse-10124-lectures


In [3]:
# @title Recurrent Neural Network

import torch.nn as nn
import torch.nn.functional as F

class SLM(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(512, 128)
        self.rnn = nn.RNN(128, 64)
        self.output = nn.Linear(64, 512)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.rnn(x)
        logits = self.output(x)
        return logits

In [4]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

def train_slm(dataset, epochs=5, batch_size=64, lr=3e-4, grad_clip=1.0):
    device = dataset.device
    V = len(dataset.tokenizer.vocab)

    model = SLM(vocab_size=V).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=dataset.collate,
    )

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss_sum = 0.0   # sum over tokens
        total_tokens = 0

        for X, Y_onehot, mask in loader:
            # X: (B,T) Long
            # Y_onehot: (B,T,V) float
            # mask: (B,T) bool

            opt.zero_grad(set_to_none=True)

            logits, _ = model(X)                    # (B,T,V)
            log_probs = F.log_softmax(logits, dim=-1)

            # per-position CE: -(y · log p)
            per_pos_loss = -(Y_onehot * log_probs).sum(dim=-1)   # (B,T)

            # ignore padding positions
            per_pos_loss = per_pos_loss * mask.float()           # (B,T)

            loss_sum = per_pos_loss.sum()                        # scalar
            n_tokens = mask.sum().item()

            if n_tokens == 0:
                continue

            loss = loss_sum / n_tokens                           # average per real token
            loss.backward()

            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            opt.step()

            total_loss_sum += loss_sum.item()
            total_tokens += n_tokens

        avg_loss = total_loss_sum / max(1, total_tokens)
        ppl = float(torch.exp(torch.tensor(avg_loss)))
        print(f"epoch {epoch:02d} | loss/token={avg_loss:.4f} | ppl={ppl:.2f}")

    return model

In [9]:
import irishGPT

r_t = irishGPT.tokenizer.Regex_Tokenizer()

dataset = irishGPT.utilities.IrishChatDataset('Datasets/zoomer.txt', r_t)

model = train_slm(dataset)

AttributeError: module 'irishGPT' has no attribute 'tokenizer'

In [ ]:
@torch.no_grad()
def generate_stateful(model, tokenizer, prompt, max_new_tokens=80,
                      temperature=1.0, top_k=None, device=None):
    """
    Uses hidden state to avoid recomputing the entire prefix every step.
    """
    model.eval()
    device = device or next(model.parameters()).device

    ids = tokenizer.encode(prompt)
    x = torch.tensor([ids], dtype=torch.long, device=device)  # (1,T)

    # 1) Prime the hidden state by running the whole prompt once
    logits, h = model(x)                      # logits: (1,T,V), h: (num_layers,1,H)

    for _ in range(max_new_tokens):
        next_logits = logits[:, -1, :]        # (1,V)
        next_id = sample_next_token_from_logits(next_logits, temperature, top_k)

        # 2) Feed ONLY the new token, carrying h forward
        x_new = torch.tensor([[next_id]], dtype=torch.long, device=device)  # (1,1)
        logits, h = model(x_new, h0=h)        # logits: (1,1,V)

        # keep a growing list for decoding
        x = torch.cat([x, x_new], dim=1)

    return tokenizer.decode(x[0].tolist())

In [ ]:
text = generate_stateful(
    model,
    dataset.tokenizer,
    prompt="hello there ",
    max_new_tokens=120,
    temperature=0.9,
    top_k=50,
    device=dataset.device,
)

print(text)